# 3. Streams and batches

Notebook 1 gave `Engine.ocr` a list of pages. This one reads pages the other ways the module offers, and measures them:

- **a stream**: `submit` a page, or `submit_file` a file the engine's workers decode, and take each result with `next`, in the order given, while more pages go in;
- **errors**: a page that fails raises its error at `next`, and the stream goes on;
- **two threads**: one submits while another takes the results;
- **the batched pipeline**: `carbon_ocr.Pipeline` runs every model on batches drawn from all the pages in flight, with the same results as the engine on the CPU;
- **throughput**: pages a second each way, and the pipeline's batch sizes.

In [1]:
import threading
import time
from pathlib import Path

import carbon_ocr
import helpers
import sample_pages

SET = helpers.model_set()
pages = sample_pages.make_all("data")
images = {name: carbon_ocr.load_image(path) for name, path in pages.items()}
engine = carbon_ocr.Engine(SET)
print(engine.workers, "workers")

10 workers


## A stream: `submit` and `next`

`submit(page)` gives the engine a page and returns its number, from 0 in the order given. The page starts as soon as a worker is free. `next()` returns the next page's result in that order, waiting for it, and `None` once every page given is out. The engine holds the array, without copying it, until `next` has returned its page. `submit` waits while two pages a worker are being read, so a long stream never holds more than that in memory.

In [2]:
names = [n for n in pages if n != "newspaper"]
t = time.perf_counter()
numbers = [engine.submit(images[n]) for n in names]
print("submitted", numbers, f"in {(time.perf_counter() - t) * 1000:.0f} ms")
for name in names:
    page = engine.next()
    print(f"{time.perf_counter() - t:6.3f} s  {name:12} {len(page['lines']):3} lines, {page['ms']['total']:5.1f} ms in the engine")
print(engine.next())  # every page given is out

submitted [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10] in 0 ms


 0.300 s  letter        21 lines, 299.7 ms in the engine
 0.301 s  columns       44 lines, 248.6 ms in the engine
 0.301 s  table         48 lines, 300.4 ms in the engine
 0.301 s  mixed         32 lines, 256.1 ms in the engine
 0.301 s  banner        14 lines, 165.4 ms in the engine
 0.301 s  receipt       57 lines,  80.8 ms in the engine
 0.303 s  slide          7 lines,  81.0 ms in the engine
 0.392 s  rotated       21 lines, 211.5 ms in the engine
 0.465 s  upside-down   21 lines, 272.1 ms in the engine
 0.465 s  skewed        21 lines, 262.7 ms in the engine
 0.466 s  scan          21 lines, 217.2 ms in the engine
None


The results come back in the order the pages went in, each as soon as it and every page before it are done.

## Files the workers decode: `submit_file`

`submit_file(path)` hands the engine a file instead of an array. The worker that takes the page reads and decodes it, beside the other pages' work, and the page's `ms["load"]` gives that time. A program that reads a folder of scans then never decodes in Python, and never holds more than the pages in flight.

In [3]:
t = time.perf_counter()
for name in names:
    engine.submit_file(pages[name])
while (page := engine.next()) is not None:
    print(f"{page['width']:5} x {page['height']:<5} decoded in {page['ms']['load']:5.1f} ms, read in {page['ms']['total']:5.1f} ms")
print(f"{len(names)} files in {time.perf_counter() - t:.2f} s")

 1700 x 2200  decoded in   7.1 ms, read in 287.4 ms
 1700 x 2200  decoded in   8.0 ms, read in 202.9 ms
 1700 x 2200  decoded in   7.1 ms, read in 186.9 ms
 2550 x 3300  decoded in  16.8 ms, read in 221.7 ms
 1700 x 2200  decoded in   7.1 ms, read in 137.9 ms
  560 x 3000  decoded in   3.4 ms, read in  89.4 ms
 1920 x 1080  decoded in   3.4 ms, read in  87.4 ms
 2200 x 1700  decoded in   5.6 ms, read in 166.0 ms
 1700 x 2200  decoded in   5.0 ms, read in 163.9 ms
 1700 x 2200  decoded in   8.5 ms, read in 153.1 ms
 1700 x 2200  decoded in  17.3 ms, read in 193.5 ms
11 files in 0.41 s


## A page that fails

A file that is not an image, or that cannot be opened, is its page's error: `next` raises it in that page's turn, and the pages after it are read as ever.

In [4]:
Path("data/not-an-image.png").write_bytes(b"this is not a PNG")
for path in ["data/letter.png", "data/not-an-image.png", "data/missing.png", "data/table.png"]:
    engine.submit_file(path)
while True:
    try:
        page = engine.next()
    except RuntimeError as e:
        print("error:", e)
        continue
    if page is None:
        break
    print("read:", page["lines"][0]["text"])

read: Notes on the Harbour Survey
error: data/not-an-image.png: the image could not be read: unknown image type
error: cannot open data/missing.png


read: Ferry passengers by quarter, 2025


## One thread submits, another takes

A stream may be fed by one thread while another takes its results, so reading files, a queue or a socket never waits on the engine's work. `next` returns `None` whenever every page given so far is out, so the taker should only ask for pages it knows were given: here a semaphore counts them.

In [5]:
files = [pages[n] for n in names] * 3
given = threading.Semaphore(0)


def feed():
    for f in files:
        engine.submit_file(f)  # waits while the engine has its fill of pages in flight
        given.release()


t = time.perf_counter()
feeder = threading.Thread(target=feed)
feeder.start()
lines = 0
for _ in files:
    given.acquire()  # a page was given, so next() waits for it rather than return None
    lines += len(engine.next()["lines"])
feeder.join()
s = time.perf_counter() - t
print(f"{len(files)} pages, {lines} lines in {s:.2f} s: {len(files) / s:.1f} pages a second")

33 pages, 921 lines in 1.05 s: 31.3 pages a second


`next(markdown=True)` and `next(html=True)` give each page's markdown or HTML as it comes out of the stream, so a program can write each page's file and drop the page.

In [6]:
out = Path("out")
out.mkdir(exist_ok=True)
for name in ["letter", "table", "banner"]:
    engine.submit_file(pages[name])
for name in ["letter", "table", "banner"]:
    page = engine.next(markdown=True)
    (out / f"{name}.md").write_text(page["markdown"], encoding="utf-8")
    print(f"out/{name}.md: {len(page['markdown'])} characters")
print((out / "table.md").read_text(encoding="utf-8")[:500])

out/letter.md: 1389 characters
out/table.md: 1102 characters
out/banner.md: 550 characters
## Ferry passengers by quarter, 2025

Passenger numbers rose in every quarter on every route. The third quarter, with the summer timetable, carried a third of the year's passengers.

<table>
<tr><td>Route</td><td></td><td colspan="2">Passengers (thousands)</td><td></td><td>Year</td></tr>
<tr><td></td><td>Q1</td><td>Q2</td><td>Q3</td><td>Q4</td><td></td></tr>
<tr><td>Harbour to Northgate</td><td>41.2</td><td>44.8</td><td>58.3</td><td>39.6</td><td>183.9</td></tr>
<tr><td>Harbour to Saltmarsh</td><


## The batched pipeline: `Pipeline`

`carbon_ocr.Pipeline` reads pages differently. Each model is a stage of its own, and each stage runs on batches drawn from every page in flight: up to `page_batch` pages an orientation or detection run, up to `line_batch` lines a recognition run, from any of the pages. That is how a GPU is kept busy, and the same stages run on the CPU (`device="cpu"`), where every page comes out as the engine gives it.

| option | default | what |
|---|---|---|
| `device` | `None`: the GPU where the build has CUDA and TensorRT, else the CPU | `"cpu"` or `"cuda"` |
| `page_batch` | 8 | pages an orientation or detection run holds at most |
| `line_batch` | 256 | lines a recognition run holds at most |
| `in_flight` | 32 | pages in the pipeline at once; `submit` waits past it |
| `threads`, `model_threads` | 0, 0 | the CPU threads preparing and assembling pages, and a CPU model run's threads (0: the physical cores) |
| `rec`, `order_model`, `precision`, `gate`, `block_order`, `table_spans` | as `Engine`'s | |
| `engine_cache` | `None` | where TensorRT keeps the engines it builds, for the next run |

In [7]:
pipeline = carbon_ocr.Pipeline(SET, device="cpu")
batch = [images[n] for n in pages]
by_pipeline = pipeline.ocr(batch)
by_engine = engine.ocr(batch)
print("device:", pipeline.device)
print("the same text as the engine's on every page:", all(a["text"] == b["text"] for a, b in zip(by_pipeline, by_engine)))
print("runs:", pipeline.stats)

device: cpu
the same text as the engine's on every page: True
runs: {'pages': 12, 'lines': 1114, 'orient_runs': 4, 'detect_runs': 7, 'recognize_runs': 52}


`stats` counts the pages and lines it read and the model runs that held them: far fewer runs than pages, since each orientation or detection run holds several pages and each recognition run many lines. The pipeline is a stream as well: `submit` copies a page in, `next` returns results in order, and `close` says no more pages are coming, after which `next` returns `None` once every page is out.

In [8]:
stream = carbon_ocr.Pipeline(SET, device="cpu")
for name in ["letter", "receipt", "slide"]:
    stream.submit(images[name])
stream.close()
while (page := stream.next()) is not None:
    print(page["lines"][0]["text"])

Notes on the Harbour Survey
HARBOUR CAFE
Channel depths, 2026


## Throughput

How many pages a second each reader gets depends on the pages and the machine. Here, on 33 sample pages (every page but the newspaper, three times), each reader takes the same pages in turns, three times over, and the best of its three is shown.

In [9]:
work = [images[n] for n in names] * 3
readers = {
    "Engine": engine,
    "Pipeline, page_batch=8, in_flight=32": pipeline,
    "Pipeline, page_batch=1, in_flight=8": carbon_ocr.Pipeline(SET, device="cpu", page_batch=1, in_flight=8),
    "Pipeline, line_batch=64": carbon_ocr.Pipeline(SET, device="cpu", line_batch=64),
}
best = {label: 0.0 for label in readers}
for reader in readers.values():
    reader.ocr(work[:4])
for _ in range(3):
    for label, reader in readers.items():
        t = time.perf_counter()
        reader.ocr(work)
        best[label] = max(best[label], len(work) / (time.perf_counter() - t))
for label, rate in best.items():
    print(f"{label:38} {rate:5.1f} pages a second")

Engine                                  30.4 pages a second
Pipeline, page_batch=8, in_flight=32    15.9 pages a second
Pipeline, page_batch=1, in_flight=8     19.4 pages a second
Pipeline, line_batch=64                 18.3 pages a second


On a CPU the engine's layout, a pinned worker per core and each page's lines spread as tasks, is the faster of the two. The pipeline's batches pay off on a GPU, where one run of many pages or lines costs little more than a run of one.

## The GPU

On an NVIDIA GPU, the pipeline runs every model on TensorRT, from the set's FP16 files by default, with the lines of all the pages in flight cropped, recognized and decoded on the device. That needs a build with CUDA and TensorRT. This Python module is built without TensorRT for now. On the GPU, use the command line (`carbon-ocr ocr --device cuda`) or the server (`carbon-ocr serve --device cuda`), both in notebook 6.

In [10]:
print("this module has the CUDA kernels:", carbon_ocr.HAS_CUDA)
try:
    carbon_ocr.Pipeline(SET, device="cuda")
except RuntimeError as e:
    print('Pipeline(device="cuda"):', e)

this module has the CUDA kernels: False
Pipeline(device="cuda"): the GPU pipeline needs CUDA and TensorRT, not built in
